# **TASK 1**

In [1]:
import numpy as np

def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    causal_mask: bool = False
) -> tuple[np.ndarray, np.ndarray]:
    """
    Computes Scaled Dot-Product Attention for 4D Multi-Head Tensors.

    Parameters
    ----------
    Q : np.ndarray
        Query tensor of shape (batch_size, num_heads, seq_len_q, d_k)
    K : np.ndarray
        Key tensor of shape (batch_size, num_heads, seq_len_k, d_k)
    V : np.ndarray
        Value tensor of shape (batch_size, num_heads, seq_len_k, d_v)
    causal_mask : bool, optional
        If True, applies upper-triangular mask (-inf) to prevent look-ahead attention.

    Returns
    -------
    output : np.ndarray
        Context vectors of shape (batch_size, num_heads, seq_len_q, d_v)
    attention_weights : np.ndarray
        Normalized attention probability weights (batch_size, num_heads, seq_len_q, seq_len_k)
    """
    # 1. Extract feature dimension d_k for scaling factor
    d_k = Q.shape[-1]
    scaling_factor = np.sqrt(d_k)

    # 2. Vectorized 4D Dot-Product: (B, H, S_q, d_k) @ (B, H, d_k, S_k) -> (B, H, S_q, S_k)
    # np.swapaxes swaps the last two dimensions of K to form K^T
    K_T = np.swapaxes(K, -2, -1)
    scores = np.matmul(Q, K_T) / scaling_factor

    # 3. Apply Causal Masking (if enabled)
    if causal_mask:
        seq_len_q, seq_len_k = Q.shape[-2], K.shape[-2]
        # Generate upper-triangular boolean matrix (k=1 excludes main diagonal)
        mask = np.triu(np.ones((seq_len_q, seq_len_k), dtype=bool), k=1)
        # Broadcast mask across batch and heads; set look-ahead entries to -1e9 (-inf equivalent)
        scores = np.where(mask, -1e9, scores)

    # 4. Numerically Stable Softmax across the Key Sequence Dimension (axis=-1)
    # Subtract row maximums to ensure exp() inputs are <= 0
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    # 5. Aggregate Values: (B, H, S_q, S_k) @ (B, H, S_k, d_v) -> (B, H, S_q, d_v)
    output = np.matmul(attention_weights, V)

    return output, attention_weights

In [2]:
batch_size = 2
num_heads = 4
seq_len = 5
d_k = 32
d_v = 32

np.random.seed(42)
Q = np.random.randn(batch_size, num_heads, seq_len, d_k)
K = np.random.randn(batch_size, num_heads, seq_len, d_k)
V = np.random.randn(batch_size, num_heads, seq_len, d_v)

In [3]:
# Execute Causal Attention
output, attn_weights = scaled_dot_product_attention(Q, K, V, causal_mask=True)

# Print Output Specifications
print("=== Tensor Shape Checks ===")
print(f"Query (Q) Shape:             {Q.shape}")
print(f"Key (K) Shape:               {K.shape}")
print(f"Value (V) Shape:             {V.shape}")
print(f"Attention Output Shape:      {output.shape}")
print(f"Attention Weights Shape:     {attn_weights.shape}\n")

# Verify causal mask (upper triangular elements should be 0.0)
print("=== Attention Weights Matrix (Batch 0, Head 0) ===")
print(np.round(attn_weights[0, 0], 4))

=== Tensor Shape Checks ===
Query (Q) Shape:             (2, 4, 5, 32)
Key (K) Shape:               (2, 4, 5, 32)
Value (V) Shape:             (2, 4, 5, 32)
Attention Output Shape:      (2, 4, 5, 32)
Attention Weights Shape:     (2, 4, 5, 5)

=== Attention Weights Matrix (Batch 0, Head 0) ===
[[1.     0.     0.     0.     0.    ]
 [0.4653 0.5347 0.     0.     0.    ]
 [0.1812 0.5281 0.2908 0.     0.    ]
 [0.2773 0.3139 0.1272 0.2816 0.    ]
 [0.2613 0.0903 0.3817 0.1147 0.152 ]]
